# 1. SETUP INICIAL

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, lower, regexp_replace, lit

spark = SparkSession.builder \
    .appName("Limpieza Inicial ONSV") \
    .getOrCreate()

# 2. CARGA DE DATOS DESDE GCS (/raw/)

In [ ]:
bucket = "sutran-bucket"
input_path = f"gs://{bucket}/raw/"
output_path = f"gs://{bucket}/trusted/"

df_personas = spark.read.option("header", True).option("encoding", "ISO-8859-1").csv(f"{input_path}BBDD_ONSV-PERSONAS_2021-2023.csv")
df_vehiculos = spark.read.option("header", True).option("encoding", "ISO-8859-1").csv(f"{input_path}BBDD_ONSV-VEHICULOS_2021-2023.csv")
df_siniestros = spark.read.option("header", True).option("encoding", "ISO-8859-1").csv(f"{input_path}BBDD_ONSV-SINIESTROS_2021-2023.csv")

# 3. LIMPIEZA ESPECÍFICA POR DATASET

## PERSONAS

In [8]:
df_personas_clean = df_personas \
    .drop("LUGAR_ATENCION_LESIONADO", "LUGAR_DE_DEFUNCION", "SE_SOMETIO_A_DOSAJE_ETILICO_CUALITATIVO") \
    .filter(~(col("EDAD") == "No indica")) \
    .withColumn("EDAD", when(col("EDAD") == "No indica", None).otherwise(col("EDAD"))) \
    .dropna(how="all")  # Elimina registros completamente vacíos

## Vehiculos

In [9]:
df_vehiculos_clean = df_vehiculos \
    .drop("ELEMENTO_TRANSPORTADO", "AMBITO_SERVICIO") \
    .dropna(how="all")

## Siniestros

In [10]:
df_siniestros_clean = df_siniestros \
    .drop("EXISTE_SENAL_VERTICAL", "CLASIFICACION_DE_LA_SENAL_VERTICAL_N_1", 
          "CLASIFICACION_DE_LA_SENAL_VERTICAL_N_2", "EXISTE_SENAL_HORIZONTAL") \
    .dropna(how="all")

# 4. LIMPIEZA COMÚN Y FORMATEO

In [13]:
def limpiar_columnas(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, c.strip().lower().replace(" ", "_").replace("á", "a").replace("é", "e")
                                                .replace("í", "i").replace("ó", "o").replace("ú", "u")
                                                .replace("ñ", "n"))
    return df

df_personas_clean = limpiar_columnas(df_personas_clean)
df_vehiculos_clean = limpiar_columnas(df_vehiculos_clean)
df_siniestros_clean = limpiar_columnas(df_siniestros_clean)

# 5. GUARDAR EN GCS (/trusted/)

In [14]:
df_personas_clean.write.mode("overwrite").option("header", True).csv(f"{output_path}personas/")
df_vehiculos_clean.write.mode("overwrite").option("header", True).csv(f"{output_path}vehiculos/")
df_siniestros_clean.write.mode("overwrite").option("header", True).csv(f"{output_path}siniestros/")

print("✅ Archivos limpios guardados en /trusted/")

✅ Archivos limpios guardados en /trusted/
